Imports and Filepaths

In [1]:
import pandas as pd
import numpy as np

market_raw_path = "/content/india-vahan-registrations-by-vehicle-category-and-fuel-fy2011-fy2025-tdc-formatted-4mae2h.csv"

Build india_ev_market

In [2]:
market_df = pd.read_csv(market_raw_path)

# EV passenger cars
ev_cars = market_df[
    (market_df["MODE"] == "Cars") &
    (market_df["FUEL_TYPE"] == "EV")
].copy()

# Extract year
ev_cars["year"] = (
    ev_cars["TIME_PERIOD"]
    .str.replace("FY", "", regex=False)
    .astype(int)
)

# Rename columns
ev_cars = ev_cars.rename(columns={
    "GEO": "country",
    "TIME_PERIOD": "financial_year",
    "MODE": "vehicle_type",
    "FUEL_TYPE": "fuel_type",
    "OBS_VALUE": "ev_registrations",
    "UNIT_MEASURE": "unit"
})

# Total passenger-car registrations
cars = market_df[market_df["MODE"] == "Cars"].copy()

total_cars = (
    cars.groupby("TIME_PERIOD")["OBS_VALUE"]
    .sum()
    .reset_index(name="total_car_registrations")
    .rename(columns={"TIME_PERIOD": "financial_year"})
)

# Combine
india_ev_market = ev_cars.merge(
    total_cars,
    on="financial_year",
    how="left"
)

# Sort
india_ev_market = india_ev_market.sort_values("year").reset_index(drop=True)

# YoY growth
india_ev_market["yoy_growth_pct"] = (
    india_ev_market["ev_registrations"].pct_change() * 100
)

# EV penetration
india_ev_market["ev_penetration_pct"] = (
    india_ev_market["ev_registrations"]
    / india_ev_market["total_car_registrations"]
    * 100
)

# Final columns
india_ev_market = india_ev_market[
    [
        "year",
        "financial_year",
        "country",
        "vehicle_type",
        "fuel_type",
        "ev_registrations",
        "total_car_registrations",
        "yoy_growth_pct",
        "ev_penetration_pct"
    ]
]

Validate market dataset

In [3]:
assert len(india_ev_market) == 15
assert india_ev_market["year"].is_monotonic_increasing
assert india_ev_market["ev_registrations"].notna().all()
assert india_ev_market["total_car_registrations"].notna().all()
assert india_ev_market.duplicated().sum() == 0

print("Market dataset validated.")
print(india_ev_market)

Market dataset validated.
    year financial_year country vehicle_type fuel_type  ev_registrations  \
0   2011         FY2011     IND         Cars        EV             411.0   
1   2012         FY2012     IND         Cars        EV             698.0   
2   2013         FY2013     IND         Cars        EV             229.0   
3   2014         FY2014     IND         Cars        EV             451.0   
4   2015         FY2015     IND         Cars        EV             634.0   
5   2016         FY2016     IND         Cars        EV             782.0   
6   2017         FY2017     IND         Cars        EV             769.0   
7   2018         FY2018     IND         Cars        EV            1213.0   
8   2019         FY2019     IND         Cars        EV            1841.0   
9   2020         FY2020     IND         Cars        EV            3215.0   
10  2021         FY2021     IND         Cars        EV            5901.0   
11  2022         FY2022     IND         Cars        EV        

Save market dataset

In [4]:
india_ev_market.to_csv(
    "/content/india_ev_market.csv",
    index=False
)

print("Saved: india_ev_market.csv")

Saved: india_ev_market.csv


Competitor dataset

In [5]:
competitor_data = [
    # Tata Motors
    {
        "financial_year": "FY2022",
        "year": 2022,
        "company": "Tata Motors",
        "ev_sales": 19106,
        "metric_definition": "EV sales (domestic PV)",
        "source": "Tata Motors Q4 FY22 Sales Release",
        "source_url": "https://www.tatamotors.com/press-releases/tata-motors-registered-total-sales-of-243459-units-in-q4-fy22/"
    },
    {
        "financial_year": "FY2023",
        "year": 2023,
        "company": "Tata Motors",
        "ev_sales": 50043,
        "metric_definition": "EV sales (IB + Domestic)",
        "source": "Tata Motors Q4 FY23 Sales Release",
        "source_url": "https://www.tatamotors.com/press-releases/tata-motors-registers-total-sales-of-251822-units-in-q4-fy23-up-3-over-q4-fy22/"
    },
    {
        "financial_year": "FY2024",
        "year": 2024,
        "company": "Tata Motors",
        "ev_sales": 73833,
        "metric_definition": "EV sales (IB + Domestic)",
        "source": "Tata Motors Q4 FY24 Sales Release",
        "source_url": "https://www.tatamotors.com/wp-content/uploads/2024/04/Tata-Motors-Sales-Release-Q4-FY24.pdf"
    },
    {
        "financial_year": "FY2025",
        "year": 2025,
        "company": "Tata Motors",
        "ev_sales": 64276,
        "metric_definition": "EV sales (IB + Domestic)",
        "source": "Tata Motors Q4 FY25 Sales Release",
        "source_url": "https://static-assets.tatamotors.com/Production/www-tatamotors-com-NEW/wp-content/uploads/2025/04/Tata-Motors-Sales-Release-Q4-FY25.pdf"
    },

    # Mahindra
    {
        "financial_year": "FY2023",
        "year": 2023,
        "company": "Mahindra",
        "ev_sales": 2416,
        "metric_definition": "Electric 4-wheeler sales",
        "source": "Mahindra FY24 Annual Report",
        "source_url": "https://www.mahindra.com/sites/default/files/2024-06/MM-Annual-Report-2023-24.pdf"
    },
    {
        "financial_year": "FY2024",
        "year": 2024,
        "company": "Mahindra",
        "ev_sales": 8025,
        "metric_definition": "Electric 4-wheeler sales",
        "source": "Mahindra FY24 Annual Report",
        "source_url": "https://www.mahindra.com/sites/default/files/2024-06/MM-Annual-Report-2023-24.pdf"
    },
    {
        "financial_year": "FY2025",
        "year": 2025,
        "company": "Mahindra",
        "ev_sales": 14183,
        "metric_definition": "Electric 4-wheeler sales",
        "source": "Mahindra FY25 Annual Report",
        "source_url": "https://www.mahindra.com/annual-report-FY2025/"
    },

    # Hyundai
    {
        "financial_year": "FY2024",
        "year": 2024,
        "company": "Hyundai",
        "ev_sales": 2120,
        "metric_definition": "EV sales",
        "source": "Hyundai FY25 Q4 Investor Presentation",
        "source_url": "https://www.hyundai.com/content/dam/hyundai/in/en/data/investor-relations/quaterly-financials/q4-investor-presentation.pdf"
    },
    {
        "financial_year": "FY2025",
        "year": 2025,
        "company": "Hyundai",
        "ev_sales": 3969,
        "metric_definition": "EV sales",
        "source": "Hyundai FY25 Q4 Investor Presentation",
        "source_url": "https://www.hyundai.com/content/dam/hyundai/in/en/data/investor-relations/quaterly-financials/q4-investor-presentation.pdf"
    }
]

competitor_ev = pd.DataFrame(competitor_data)

competitor_ev = competitor_ev.sort_values(
    ["company", "year"]
).reset_index(drop=True)

competitor_ev["yoy_growth_pct"] = (
    competitor_ev.groupby("company")["ev_sales"].pct_change() * 100
)

competitor_ev["data_quality"] = "official_company_source"

competitor_ev = competitor_ev[
    [
        "year",
        "financial_year",
        "company",
        "ev_sales",
        "yoy_growth_pct",
        "metric_definition",
        "data_quality",
        "source",
        "source_url"
    ]
]

Validate and Save

In [6]:
assert competitor_ev.duplicated().sum() == 0
assert competitor_ev["ev_sales"].notna().all()
assert competitor_ev["company"].isin(
    ["Tata Motors", "Mahindra", "Hyundai"]
).all()

print("Competitor dataset validated.")
print(competitor_ev)

competitor_ev.to_csv(
    "/content/competitor_ev_sales.csv",
    index=False
)

print("\nSaved: competitor_ev_sales.csv")

Competitor dataset validated.
   year financial_year      company  ev_sales  yoy_growth_pct  \
0  2024         FY2024      Hyundai      2120             NaN   
1  2025         FY2025      Hyundai      3969       87.216981   
2  2023         FY2023     Mahindra      2416             NaN   
3  2024         FY2024     Mahindra      8025      232.160596   
4  2025         FY2025     Mahindra     14183       76.735202   
5  2022         FY2022  Tata Motors     19106             NaN   
6  2023         FY2023  Tata Motors     50043      161.922956   
7  2024         FY2024  Tata Motors     73833       47.539116   
8  2025         FY2025  Tata Motors     64276      -12.944076   

          metric_definition             data_quality  \
0                  EV sales  official_company_source   
1                  EV sales  official_company_source   
2  Electric 4-wheeler sales  official_company_source   
3  Electric 4-wheeler sales  official_company_source   
4  Electric 4-wheeler sales  official_c